# Rate Limiters API Reference

Developer-facing statements defined in `libs/core/langchain_core/rate_limiters.py`.

# `BaseRateLimiter: abc.ABC`

Abstract base class for synchronous and asynchronous rate limiters.

Concrete subclasses must implement both `acquire` and `aacquire`. The abstract methods do not provide executor wrappers, timeout handling, or explicit `NotImplementedError` behavior.

## Required subclass hooks

### `acquire`

Attempts to acquire the tokens required by the limiter.

```python
@abc.abstractmethod
acquire(
    self,
    *,
    blocking: bool = True, # Whether to wait until the required tokens become available
) -> bool # True when tokens are acquired; otherwise False
```

When `blocking` is `False`, implementations should return the result immediately.

### `aacquire`

Asynchronously attempts to acquire the tokens required by the limiter.

```python
@abc.abstractmethod
async aacquire(
    self,
    *,
    blocking: bool = True, # Whether to wait until the required tokens become available
) -> bool # True when tokens are acquired; otherwise False
```

When `blocking` is `False`, implementations should return the result immediately.

---

# `InMemoryRateLimiter: BaseRateLimiter`

Thread-safe, in-process rate limiter based on a token-bucket algorithm.

The limiter controls request frequency using time-based tokens; these are unrelated to LLM input or output tokens. It cannot coordinate limits across processes and does not account for request size.

## Fields

```python
requests_per_second: float # Rate at which tokens are added to the bucket
available_tokens: float = 0.0 # Tokens currently available for requests
max_bucket_size: float # Maximum number of accumulated tokens
last: float | None = None # Monotonic timestamp of the previous refill calculation
check_every_n_seconds: float # Delay between blocking acquisition attempts
```

## Constructor

```python
InMemoryRateLimiter(
    *,
    requests_per_second: float = 1, # Number of request tokens added per second
    check_every_n_seconds: float = 0.1, # Delay between checks while waiting for a token
    max_bucket_size: float = 1, # Maximum accumulated tokens and resulting burst capacity
) -> None
```

`max_bucket_size` is documented as requiring a value of at least `1`; the constructor does not explicitly validate it.

The bucket begins with no available tokens. Each successful acquisition consumes one token.

## Methods

### `acquire`

Attempts to acquire one token synchronously.

```python
acquire(
    self,
    *,
    blocking: bool = True, # Whether to keep checking until a token becomes available
) -> bool # True when a token is acquired; otherwise False
```

With `blocking=False`, it performs one immediate acquisition attempt. With `blocking=True`, it sleeps for `check_every_n_seconds` between attempts until one token is acquired.

### `aacquire`

Attempts to acquire one token asynchronously.

```python
async aacquire(
    self,
    *,
    blocking: bool = True, # Whether to keep checking until a token becomes available
) -> bool # True when a token is acquired; otherwise False
```

With `blocking=False`, it performs one immediate acquisition attempt. With `blocking=True`, it awaits `asyncio.sleep(check_every_n_seconds)` between attempts until one token is acquired.

## Behaviour

Token refill calculations use `time.monotonic`. Accumulated tokens are capped at `max_bucket_size`, allowing bursts only up to that capacity. Access to token consumption is protected by a thread lock.

In [ ]:
import asyncio # Import utilities for asynchronous execution
import time # Import timing utilities

from langchain_core.rate_limiters import InMemoryRateLimiter # Import LangChain's in-memory rate limiter


sync_limiter = InMemoryRateLimiter( # Create a synchronous rate limiter
    requests_per_second=2, # Allow two request tokens to be generated per second
    check_every_n_seconds=0.1, # Check every 0.1 seconds while waiting
    max_bucket_size=1, # Allow at most one unused token to accumulate
) # Finish creating the synchronous limiter

immediate_result = sync_limiter.acquire(blocking=False) # Attempt to acquire a token without waiting
print("Immediate acquisition:", immediate_result) # Display whether a token was immediately available

for request_number in range(1, 4): # Simulate three rate-limited requests
    sync_limiter.acquire() # Wait until one request token becomes available
    print(f"Sending synchronous request {request_number}") # Simulate sending the permitted request
    print("Time:", round(time.perf_counter(), 2)) # Display when the request was permitted


async def send_async_requests() -> None: # Define an asynchronous rate-limited workflow
    async_limiter = InMemoryRateLimiter( # Create a separate asynchronous rate limiter
        requests_per_second=2, # Allow two request tokens to be generated per second
        check_every_n_seconds=0.1, # Check asynchronously every 0.1 seconds
        max_bucket_size=1, # Allow at most one unused token to accumulate
    ) # Finish creating the asynchronous limiter

    for request_number in range(1, 4): # Simulate three asynchronous requests
        await async_limiter.aacquire() # Wait asynchronously until one token becomes available
        print(f"Sending asynchronous request {request_number}") # Simulate sending the permitted request
        print("Time:", round(time.perf_counter(), 2)) # Display when the request was permitted


asyncio.run(send_async_requests()) # Run the asynchronous rate-limited workflow